In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
try:
    from tensorflow.keras.layers import TextVectorization            # TF ≥2.6
except ImportError:
    from tensorflow.keras.layers.experimental.preprocessing import TextVectorization  # TF <2.6
from tensorflow.keras import layers, models
print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

In [ ]:
TRAIN_FILE = "train-data.tsv"
TEST_FILE  = "valid-data.tsv"

# Chargement du jeu d’entraînement
train_df = pd.read_csv(
    TRAIN_FILE,               # <-- on lit bien le fichier d’entraînement
    sep="\t",                 # séparateur tabulation
    header=None,              # pas d’en‑tête dans le fichier source
    names=["type", "msg"]     # on nomme les colonnes
).dropna()                    # on enlève les éventuelles lignes vides

# Aperçu rapide pour vérifier
print("Aperçu du jeu d’entraînement :")
display(train_df.head())      # montre les 5 premières lignes

print(f"\nNombre total de messages dans le train set : {len(train_df):,}")

In [ ]:
test_df = pd.read_csv(
    TEST_FILE,                 # <-- on lit le fichier de validation
    sep="\t",                  # séparateur tabulation
    header=None,               # pas d’en‑tête
    names=["type", "msg"]      # noms de colonnes
).dropna()                     # suppression des lignes vides, le cas échéant

# Aperçu rapide pour contrôle
print("Aperçu du jeu de validation :")
display(test_df.head())        # affiche les 5 premières lignes

print(f"\nNombre total de messages dans le test set : {len(test_df):,}")

In [ ]:
# === Encodage des étiquettes + aperçu =========================
train_df["type"] = pd.factorize(train_df["type"])[0]
test_df["type"]  = pd.factorize(test_df["type"])[0]

# Visualiser les premières lignes pour contrôle
train_df.head()


In [ ]:
# === Construction du jeu d’entraînement TensorFlow ===========
train_labels = train_df["type"].values   # vecteur numpy des étiquettes

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["msg"].values, train_labels)  # couples (texte, étiquette)
)

In [ ]:
# === Construction du jeu de test TensorFlow ====================
test_labels = test_df["type"].values     # étiquettes du jeu de validation

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["msg"].values, test_labels)  # couples (texte, étiquette)
)

# Affiche la spécification des éléments pour vérifier la forme attendue
print("\nSpécification des éléments du jeu de test :")
print(test_ds.element_spec)


In [ ]:
# === Préparation des jeux de données pour l’entraînement ===============

BUFFER_SIZE = 100      # taille du buffer pour le shuffle
BATCH_SIZE  = 32       # taille d'un batch

# Jeu d'entraînement : mélange, batch, puis pré‑lecture asynchrone
train_ds = (
    train_ds
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Jeu de validation : batch puis pré‑lecture asynchrone (pas de shuffle)
test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Jeu d’entraînement →", train_ds)
print("Jeu de validation  →", test_ds)


In [ ]:
# === Étape 4 : vectorisation du texte ==================================
# On crée puis on « adapte » (entraîne) la couche TextVectorization sur
# le corpus d’entraînement pour transformer chaque SMS en séquence
# d’entiers.

VOCAB_SIZE        = 1000        # nombre max de tokens conservés
MAX_SEQUENCE_LEN  = 1000        # longueur fixe de sortie (padding/tronc.)

vec = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LEN,
)

print("▶️  Adaptation de la couche TextVectorization…")
vec.adapt(train_ds.map(lambda text, label: text))   # ⬅️ “fit” sur le texte
print(f"✅  Vocabulaire appris : {len(vec.get_vocabulary())} tokens")


In [ ]:
# === Coup d’œil sur le vocabulaire appris =============================
vocab = np.array(vec.get_vocabulary())

print("📝  Les 20 premiers tokens du vocabulaire :")
print(vocab[:20])

In [ ]:
# === Étape 🛠  |  Construction puis compilation du modèle ================

print("\n📐  Construction du réseau de neurones…")

model = tf.keras.Sequential([
    # 1️⃣  Couche de vectorisation du texte (déjà « adaptée » plus haut)
    vec,

    # 2️⃣  Embedding : chaque token → vecteur dense de taille 64
    tf.keras.layers.Embedding(
        input_dim=len(vec.get_vocabulary()),  # taille vocabulaire
        output_dim=64,
        mask_zero=True                       # ignore le padding (0)
    ),

    # 3️⃣/4️⃣  Deux couches LSTM bidirectionnelles
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True)
    ),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(32)
    ),

    # 5️⃣  Densément connectée + Dropout
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    # 6️⃣  Couche de sortie (logits) —> 1 neurone pour la classe spam/ham
    tf.keras.layers.Dense(1)
])

print(model.summary(), "\n")  # aperçu architecture

# --- Compilation -------------------------------------------------------
print("⚙️  Compilation du modèle (loss=BinaryCrossentropy, opt=Adam)…")

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=['accuracy']
)

print("✅  Modèle prêt pour l’entraînement !")


In [ ]:
# === Étape 🚀  |  Entraînement du modèle =================================
print("\n🎯  Lancement de l’entraînement…  (10 epoch)")

history = model.fit(
    train_ds,
    validation_data=test_ds,
    validation_steps=30,   # on évalue sur 30 batches de la validation
    epochs=10,
)

print("\n📊  Entraînement terminé !")
print(f"   • Perte finale (train) : {history.history['loss'][-1]:.4f}")
print(f"   • Précision finale (train) : {history.history['accuracy'][-1]:.4f}")
print(f"   • Perte finale (val)   : {history.history['val_loss'][-1]:.4f}")
print(f"   • Précision finale (val)   : {history.history['val_accuracy'][-1]:.4f}")


In [ ]:
# === Étape ✅  |  Évaluation finale sur l’ensemble de test ===============

print("\n🔎  Évaluation du modèle sur tout le jeu de validation…")
test_loss, test_acc = model.evaluate(test_ds, verbose=0)

print(f"\n📈  Résultats :")
print(f"   • Perte (loss)  : {test_loss:.4f}")
print(f"   • Précision (accuracy) : {test_acc:.4f}")


In [ ]:
# === Étape ✅  |  Fonction utilitaire pour tracer l’apprentissage ============

import matplotlib.pyplot as plt

def plot_graphs(history, metric):
    """
    Affiche l’évolution d’un indicateur (accuracy ou loss) sur les
    jeux d’entraînement et de validation au fil des époques.

    Paramètres
    ----------
    history : tf.keras.callbacks.History
        L’objet History renvoyé par model.fit().
    metric : str
        Le nom de la métrique à tracer (« accuracy », « loss », etc.).
    """
    print(f"\n📊  Affichage du graphique pour la métrique : {metric}")

    plt.figure(figsize=(6, 4))
    plt.plot(history.history[metric], label=f'{metric} (train)')
    plt.plot(history.history['val_'+metric], label=f'{metric} (val)')
    plt.xlabel("Époques")
    plt.ylabel(metric.capitalize())
    plt.title(f"{metric.capitalize()} — entraînement vs validation")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# === Étape 📈 | Visualisation des courbes d’apprentissage ===================
print("\n🔍 Visualisation des courbes d’apprentissage (accuracy & loss)…")

plt.figure(figsize=(16, 8))

# ── Courbe d’accuracy ─────────────────────────────────────────────────────
plt.subplot(1, 2, 1)
plot_graphs(history, 'accuracy')
plt.ylim(0, 1)                 # bornes plus lisibles
plt.title("Accuracy")

# ── Courbe de perte (loss) ────────────────────────────────────────────────
plt.subplot(1, 2, 2)
plot_graphs(history, 'loss')
plt.ylim(0, None)
plt.title("Loss")

plt.tight_layout()
plt.show()


In [ ]:
# === Étape 🗒️ | Inspection rapide de l’historique ==========================
print("\n📊 Récapitulatif brut des métriques enregistrées à chaque époque :")

h = history.history

print("\n• Loss (train)      :", h["loss"])
print("• Loss (validation) :", h["val_loss"])
print("• Accuracy (train)  :", h["accuracy"])
print("• Accuracy (val.)   :", h["val_accuracy"])

In [ ]:
# === Étape 🔍 | Prédiction d’un SMS =========================================
# La fonction prend un texte brut et renvoie :
#   [ score_logit , 'ham' ]  si score < 0.5
#   [ score_logit , 'spam' ] si score ≥ 0.5
# (On affiche aussi le tenseur des logits pour vérification.)

# function to predict messages based on model
# retourne [probabilité, label]  (0 = ham, 1 = spam)
def predict_message(pred_text):
    # On prépare la phrase sous forme de tenseur 1‑D
    inputs = tf.constant([pred_text])

    # Logits bruts
    logits = model.predict(inputs, verbose=0)[0][0]
    # Probabilité grâce à la sigmoïde
    prob   = tf.sigmoid(logits).numpy().item()

    print(f"Logit : {logits:.4f}   →   Prob(spam) : {prob:.4f}")
    return [prob, "ham" if prob < 0.5 else "spam"]


# Exemple de test rapide
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print("Résultat retourné :", prediction)

# Exemple de test rapide
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print("Résultat retourné :", prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460 4",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()